# Bài 2 (Trang 37, 38) - Online Retail (United Kingdom)
## Câu 12: Tìm tập phổ biến và luật kết hợp bằng FP-Growth (min_sup = 3%, min_conf = 50%), so sánh với Apriori

**Sinh viên:** Trần Đình Võ - **MSSV:** 6451071088

In [1]:
HO_TEN = "Trần Đình Võ"
MSSV = "6451071088"
print(f"Họ tên: {HO_TEN} - MSSV: {MSSV}")

Họ tên: Trần Đình Võ - MSSV: 6451071088


### Import thư viện và đọc dữ liệu

In [2]:
import time
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")
from mlxtend.frequent_patterns import apriori, fpgrowth, association_rules
%matplotlib inline

df = pd.read_excel("../Dataset/Online Retail.xlsx")

### Các bước xử lý của câu 1 → 11 (chuẩn bị dữ liệu)

In [3]:
df["Description"] = df["Description"].str.strip()
df = df.dropna(axis=0, subset=["InvoiceNo"])
df["InvoiceNo"] = df["InvoiceNo"].astype(str)

df = df[~df["InvoiceNo"].str.contains("C")]

grouped = (df[df["Country"] == "United Kingdom"]
           .groupby(["InvoiceNo", "Description"])["Quantity"]
           .sum())

basket = grouped.unstack().reset_index().fillna(0).set_index("InvoiceNo")

def encode_units(x):
    return 1 if x >= 1 else 0

# Tương đương basket.map(encode_units) nhưng vector hóa cho nhanh; kiểu bool theo khuyến nghị của mlxtend
basket_sets = basket >= 1

basket_sets = basket_sets.drop(columns=["POSTAGE"], errors="ignore")

start = time.time()
frequent_itemsets = apriori(basket_sets, min_support=0.03, use_colnames=True)
apriori_time = time.time() - start

rules = association_rules(frequent_itemsets, num_itemsets=len(basket_sets),
                          metric="confidence", min_threshold=0.5)

rules["antecedents"] = rules["antecedents"].apply(lambda x: ", ".join(sorted(x)))
rules["consequents"] = rules["consequents"].apply(lambda x: ", ".join(sorted(x)))

support = rules["support"].values
confidence = rules["confidence"].values

### Câu 12: Tìm tập phổ biến và luật kết hợp bằng FP-Growth (min_sup = 3%, min_conf = 50%), so sánh với Apriori

In [4]:
start = time.time()
fp_itemsets = fpgrowth(basket_sets, min_support=0.03, use_colnames=True)
fp_time = time.time() - start
fp_rules = association_rules(fp_itemsets, num_itemsets=len(basket_sets),
                             metric="confidence", min_threshold=0.5)
fp_rules["antecedents"] = fp_rules["antecedents"].apply(lambda x: ", ".join(sorted(x)))
fp_rules["consequents"] = fp_rules["consequents"].apply(lambda x: ", ".join(sorted(x)))

In [5]:
print("Số tập phổ biến FP-Growth:", len(fp_itemsets))
print("Số luật FP-Growth:", len(fp_rules))
for i, r in fp_rules.iterrows():
    print(f"Rule {i + 1}: {r['antecedents']} => {r['consequents']} "
          f"(support = {r['support']:.3f}, confidence = {r['confidence']:.3f})")

Số tập phổ biến FP-Growth: 131
Số luật FP-Growth: 10
Rule 1: JUMBO BAG PINK POLKADOT => JUMBO BAG RED RETROSPOT (support = 0.042, confidence = 0.677)
Rule 2: JUMBO STORAGE BAG SUKI => JUMBO BAG RED RETROSPOT (support = 0.037, confidence = 0.618)
Rule 3: JUMBO  BAG BAROQUE BLACK WHITE => JUMBO BAG RED RETROSPOT (support = 0.031, confidence = 0.626)
Rule 4: JUMBO SHOPPER VINTAGE RED PAISLEY => JUMBO BAG RED RETROSPOT (support = 0.035, confidence = 0.580)
Rule 5: ALARM CLOCK BAKELIKE RED => ALARM CLOCK BAKELIKE GREEN (support = 0.030, confidence = 0.605)
Rule 6: ALARM CLOCK BAKELIKE GREEN => ALARM CLOCK BAKELIKE RED (support = 0.030, confidence = 0.643)
Rule 7: ROSES REGENCY TEACUP AND SAUCER => GREEN REGENCY TEACUP AND SAUCER (support = 0.038, confidence = 0.732)
Rule 8: GREEN REGENCY TEACUP AND SAUCER => ROSES REGENCY TEACUP AND SAUCER (support = 0.038, confidence = 0.751)
Rule 9: PINK REGENCY TEACUP AND SAUCER => GREEN REGENCY TEACUP AND SAUCER (support = 0.031, confidence = 0.821)
Rul

In [6]:
# So sánh với Apriori
same_itemsets = set(frequent_itemsets["itemsets"]) == set(fp_itemsets["itemsets"])
key = lambda d: set(zip(d["antecedents"], d["consequents"]))
same_rules = key(rules) == key(fp_rules)
comparison = pd.DataFrame({
    "Apriori": [len(frequent_itemsets), len(rules), round(apriori_time, 3)],
    "FP-Growth": [len(fp_itemsets), len(fp_rules), round(fp_time, 3)],
}, index=["Số tập phổ biến", "Số luật", "Thời gian (s)"])
print("Tập phổ biến giống nhau:", same_itemsets)
print("Tập luật giống nhau:", same_rules)
comparison

Tập phổ biến giống nhau: True
Tập luật giống nhau: True


,Apriori,FP-Growth
Số tập phổ biến,131.000,131.000
Số luật,10.000,10.000
Thời gian (s),1.667,0.819


### Nhận xét
- FP-Growth cho ra **cùng 131 tập phổ biến và cùng 10 luật** như Apriori → hai thuật toán cho kết quả giống nhau.
- FP-Growth chạy nhanh hơn Apriori vì không phải sinh tập ứng viên và chỉ duyệt dữ liệu 2 lần để xây cây FP.